In [1]:
# Tiples EXtraction code
import spacy 
import pandas as pd
from collections import Counter
import re

# ----------------------------
# 1. Load spaCy model
# ----------------------------
nlp = spacy.load("en_core_web_sm")

# ----------------------------
# 2. Improved Triple Extraction Function (Fixed)
# ----------------------------
def clean_phrase(phrase):
    """
    Clean a phrase: Strip articles, normalize spaces, limit length.
    """
    if not phrase or len(phrase.strip()) < 2:
        return None
    # Strip leading articles/determiners
    phrase = re.sub(r'^(the|a|an)\s+', '', phrase, flags=re.IGNORECASE).strip()
    # Normalize spaces and limit to first 5 words (to avoid run-ons)
    words = phrase.split()[:5]
    phrase = ' '.join(words).strip()
    if len(phrase) < 2:
        return None
    return phrase.title()  # Title case for readability

def is_meaningful_entity(token_or_chunk):
    """
    Check if a token/chunk is a meaningful entity (proper noun, org, person, etc.).
    """
    # If it's a chunk (Span), check its first token or entities
    if hasattr(token_or_chunk, 'ents') and token_or_chunk.ents:
        return True
    # For tokens: Check POS (proper nouns, nouns) and avoid common words
    if hasattr(token_or_chunk, 'pos_') and token_or_chunk.pos_ in ('PROPN', 'NOUN'):
        text_lower = token_or_chunk.text.lower()
        if text_lower not in {
            'people', 'country', 'world', 'time', 'way', 'thing', 'man', 'woman', 'year', 'day',
            'one', 'two', 'new', 'old', 'big', 'small'
        }:
            return True
    return False

def extract_triples(doc, strict_mode=True):
    """
    Extract clean SVO triples.
    - strict_mode: If True, require both subj/obj to be meaningful entities.
    """
    triples = []
    for sent in doc.sents:
        # Get noun chunks for potential entities
        chunks = list(sent.noun_chunks)
        ents_set = {ent.text.lower() for ent in sent.ents}
        
        for token in sent:
            if token.dep_ in ("nsubj", "csubj"):
                # Extract subject tokens (filter punctuation) using chunk or subtree
                subj_tokens = []
                subj_chunk = next((chunk for chunk in chunks if token in chunk), None)
                if subj_chunk:
                    # Use chunk's tokens
                    subj_tokens = [t for t in subj_chunk if t.pos_ != "PUNCT"]
                else:
                    # Fallback to subtree tokens
                    subj_tokens = [t for t in token.subtree if t.pos_ != "PUNCT"]
                
                if not subj_tokens:
                    continue
                
                subject_raw = ' '.join([t.text for t in subj_tokens])
                subject = clean_phrase(subject_raw)
                
                if not subject or not is_meaningful_entity(token):
                    continue
                
                verb = token.head.lemma_.lower()
                
                # Skip uninformative verbs (expanded list for news data)
                skip_verbs = {
                    "be", "have", "do", "get", "take", "make", "set", "go", "come", "say", "think", "know", "see"
                }
                if verb in skip_verbs:
                    continue
                
                # Find objects
                for child in token.head.children:
                    if child.dep_ in ("dobj", "pobj"):
                        # Extract object tokens (filter punctuation) using chunk or subtree
                        obj_tokens = []
                        obj_chunk = next((chunk for chunk in chunks if child in chunk), None)
                        if obj_chunk:
                            # Use chunk's tokens
                            obj_tokens = [t for t in obj_chunk if t.pos_ != "PUNCT"]
                        else:
                            # Fallback to subtree tokens
                            obj_tokens = [t for t in child.subtree if t.pos_ != "PUNCT"]
                        
                        if not obj_tokens:
                            continue
                        
                        obj_raw = ' '.join([t.text for t in obj_tokens])
                        obj = clean_phrase(obj_raw)
                        
                        if not obj or not is_meaningful_entity(child):
                            continue
                        
                        # For strict mode: Ensure at least one is a named entity
                        if strict_mode and subject.lower() not in ents_set and obj.lower() not in ents_set:
                            continue
                        
                        # Simple quality score: Higher for proper entities, shorter phrases
                        score = 1
                        sent_ents = [e for e in sent.ents if e.text.lower() in (subject.lower(), obj.lower())]
                        if any(e.label_ in ('PERSON', 'ORG', 'GPE', 'MONEY', 'DATE') for e in sent_ents):
                            score += 2
                        if 2 <= len(subject.split()) <= 4 and 2 <= len(obj.split()) <= 4:  # Slightly looser for news
                            score += 1
                        
                        triples.append((subject, verb, obj, score))
    
    # Remove duplicates within doc (based on subj/rel/obj, keep highest score)
    unique_triples = {}
    for triple in triples:
        key = (triple[0].lower(), triple[1], triple[2].lower())
        if key not in unique_triples or triple[3] > unique_triples[key][3]:
            unique_triples[key] = triple
    return list(unique_triples.values())

# ----------------------------
# 3. Load Dataset
# ----------------------------
dataset_path = "C:/Users/HP/Downloads/cleaned_dataset.csv"
try:
    df = pd.read_csv(dataset_path)
    print(f"Loaded dataset with {len(df)} rows.")
except Exception as e:
    print(f"Error: {e}")
    exit(1)

texts = (df["headline_clean"].fillna("") + ". " + df["short_description_clean"].fillna("")).tolist()
max_texts = 10000  # Adjust as needed
texts = texts[:max_texts]
print(f"Processing {len(texts)} texts...")

# ----------------------------
# 4. Extract Triples with Debug
# ----------------------------
all_triples = []
for i, text in enumerate(texts):
    if i % 1000 == 0:
        print(f"Processed {i}/{len(texts)} texts...")
    if i < 3:  # Debug: Print raw parse for first 3 texts
        doc = nlp(text)
        print(f"\n--- Debug for text {i+1}: '{text[:100]}...' ---")
        print("Noun Chunks:", [chunk.text for chunk in doc.noun_chunks])
        print("Entities:", [(ent.text, ent.label_) for ent in doc.ents])
        print("Sample Dependencies:", [(token.text, token.dep_, token.head.text) for token in list(doc)[:10]])
    doc = nlp(text)
    all_triples.extend(extract_triples(doc, strict_mode=True))  # Set to False for more triples

print(f"\nExtracted {len(all_triples)} unique triples.")

# ----------------------------
# 5. Rank by Frequency and Quality
# ----------------------------
# Count (case-insensitive for subj/obj)
triple_counter = Counter()
for triple in all_triples:
    subj, rel, obj, score = triple
    lower_key = (subj.lower(), rel, obj.lower())
    triple_counter[lower_key] += 1  # Count freq first

# Now aggregate scores (average per unique triple)
score_aggregator = {}
for triple in all_triples:
    subj, rel, obj, score = triple
    lower_key = (subj.lower(), rel, obj.lower())
    if lower_key not in score_aggregator:
        score_aggregator[lower_key] = {'total_score': 0, 'count': 0}
    score_aggregator[lower_key]['total_score'] += score
    score_aggregator[lower_key]['count'] += 1

# Get top 100 by freq, then avg score
top_items = []
for lower_key, freq in triple_counter.most_common(100):
    subj_lower, rel, obj_lower = lower_key
    if lower_key in score_aggregator:
        avg_score = score_aggregator[lower_key]['total_score'] / score_aggregator[lower_key]['count']
    else:
        avg_score = 1.0  # Default
    top_items.append((subj_lower, rel, obj_lower, freq, round(avg_score, 2)))

# Prepare display with title case
top_triples = []
for subj_lower, rel, obj_lower, freq, avg_score in top_items:
    subj_display = subj_lower.title()
    obj_display = obj_lower.title()
    top_triples.append((subj_display, rel, obj_display, freq, avg_score))

# DataFrame
triples_df = pd.DataFrame(top_triples, columns=["Entity1", "Relation", "Entity2", "Frequency", "Quality Score"])

# ----------------------------
# 6. Save and Display
# ----------------------------
output_path = "meaningful_triples_improved.csv"
triples_df.to_csv(output_path, index=False)
print(f"\nSaved top 100 improved triples to '{output_path}'.")
print("\nTop 20 Improved Triples (sorted by Frequency, then Quality):")
print(triples_df.head(20).to_string(index=False))


Loaded dataset with 120436 rows.
Processing 10000 texts...
Processed 0/10000 texts...

--- Debug for text 1: '23 of the funniest tweets about cats and dogs this week sept 1723. until you have a dog you dont und...' ---
Noun Chunks: ['the funniest', 'cats', 'dogs', 'you', 'a dog', 'you', 'what']
Entities: [('23', 'CARDINAL'), ('this week sept 1723', 'DATE')]
Sample Dependencies: [('23', 'nsubj', 'tweets'), ('of', 'prep', '23'), ('the', 'det', 'funniest'), ('funniest', 'pobj', 'of'), ('tweets', 'nsubj', 'sept'), ('about', 'prep', 'tweets'), ('cats', 'pobj', 'about'), ('and', 'cc', 'cats'), ('dogs', 'conj', 'cats'), ('this', 'det', 'week')]

--- Debug for text 2: 'the funniest tweets from parents this week sept 1723. accidentally put grownup toothpaste on my todd...' ---
Noun Chunks: ['the funniest', 'parents', 'grownup toothpaste', 'my toddlers', 'toothbrush', 'he', 'i', 'his teeth', 'a carolina reaper', 'tabasco sauce']
Entities: [('this week sept 1723', 'DATE'), ('grownup toothpaste', 